# Experiment 02 — DSLD Purpose/Risk pilot: inspecting the 1,000 sample and the 100-label gold set

**Goal:** *see* the two artifacts the pilot produced, before any local model is chosen.

| Artifact | Path | Rows |
|---|---|---|
| Flattened labels (mechanical, no LLM) | `data/dsld/runs/20260908T125504Z/raw/pilot-1000.jsonl` | 1,000 |
| Gold extractions (`claude-opus-5`, human-directed) | `datasets/pilot/gold-100.jsonl` | 100 |

This notebook is **read-only**: it loads, checks and displays. It calls no model and
writes nothing. It exists to answer three questions:

1. What did the flattening actually produce — is the `SKIP_TYPES` fix visible in the text?
2. What did the gold labelling actually yield — how much purpose/risk, at what confidence?
3. Where is the gold set weak, and is it a fair basis for scoring a local model?

Background: [`docs/dsld-purpose-risk-findings.md`](../docs/dsld-purpose-risk-findings.md).
No `matplotlib`/`seaborn` in this environment, so everything below is tables and text.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, re, textwrap
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PILOT_JSONL   = ROOT / "data/dsld/runs/20260908T125504Z/raw/pilot-1000.jsonl"
PILOT_MANIFEST= ROOT / "data/dsld/runs/20260908T125504Z/manifests/pilot-1000.json"
GOLD_JSONL    = ROOT / "datasets/pilot/gold-100.jsonl"
GOLD_MANIFEST = ROOT / "datasets/pilot/gold-100.manifest.json"

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 120)

def read_jsonl(path):
    with open(path) as fh:
        return [json.loads(line) for line in fh if line.strip()]

for p in (PILOT_JSONL, PILOT_MANIFEST, GOLD_JSONL, GOLD_MANIFEST):
    print(f"{'OK ' if p.exists() else 'MISSING'} {p.relative_to(ROOT)}")

## 1. Provenance — what the manifests claim

Read these first. They record how the sample was drawn (seed, frame, coverage
shortfall) and how the gold set was produced. Everything later in the notebook is
only interpretable against them.

In [ ]:
pilot_mf = json.loads(PILOT_MANIFEST.read_text())
gold_mf  = json.loads(GOLD_MANIFEST.read_text())

print("=== SAMPLING FRAME " + "=" * 51)
for k in ("dsp_csv", "seed", "n_dsp_rows", "n_dsld_only_rows",
          "n_both_rows_held_back", "n_requested", "n_drawn"):
    print(f"  {k:<24} {pilot_mf[k]}")

print("\n=== DUMP COVERAGE " + "=" * 52)
ref, have = pilot_mf["n_ids_referenced"], pilot_mf["n_ids_in_dump"]
missing = pilot_mf["n_ids_missing_from_dump"]
print(f"  iDISK references          {ref:,} DSLD IDs")
print(f"  present in the dump       {have:,}")
print(f"  missing                   {missing}  ({missing / ref:.4%})")
print(f"  -> {pilot_mf['ids_missing_from_dump']}")
print("  Handover §5 asked whether the shortfall is 'a few dozen' (noise) or")
print("  'thousands' (changes what extending to the whole corpus means).")
print(f"  It is {missing}. Noise. Extending to all {pilot_mf['n_dsld_only_rows']:,} rows is not gated on this.")

print("\n=== GOLD PRODUCTION " + "=" * 50)
for k in ("model", "executed_at", "n_extracted", "n_ungrounded_entries"):
    print(f"  {k:<24} {gold_mf[k]}")
print("\n  method:")
print(textwrap.fill(gold_mf["method"], 92, initial_indent="    ", subsequent_indent="    "))
print(f"\n  errors: {gold_mf['errors'] or 'none'}")

## 2. Load both artifacts and check they line up

The gold set must be a subset of the pilot, keyed on `dsld_id`, or scoring a local
model against it later compares against passages the model never saw.

In [ ]:
pilot = read_jsonl(PILOT_JSONL)
gold  = read_jsonl(GOLD_JSONL)

pilot_by_id = {r["dsld_id"]: r for r in pilot}

print(f"pilot rows {len(pilot):>6,}   unique ids {len(pilot_by_id):>6,}")
print(f"gold  rows {len(gold):>6,}   unique ids {len({g['dsld_id'] for g in gold}):>6,}")
print(f"pilot fields  {sorted(pilot[0])}")
print(f"gold  fields  {sorted(gold[0])}")

orphans = [g["dsld_id"] for g in gold if g["dsld_id"] not in pilot_by_id]
print(f"\ngold ids absent from the pilot: {len(orphans)}  {'-> OK' if not orphans else orphans}")
print(f"gold ids agree with gold manifest: "
      f"{[str(g['dsld_id']) for g in gold] == gold_mf['dsld_ids']}")

## 3. Artifact 1 — the flattened labels

One merged passage per label (handover §3). The `[Statement Type]` prefixes are kept
deliberately: unreliable as a filter, useful as a hint to a small model.

In [ ]:
pilot_df = pd.DataFrame(
    {"dsld_id": r["dsld_id"], "name": r["name"], "chars": len(r["passage"]),
     "statements": r["passage"].count("\n[")}
    for r in pilot
)

print("passage size, characters")
print(pilot_df["chars"].describe(percentiles=[.5, .95]).round(0).to_string())
print(f"\nmanifest recorded: {pilot_mf['passage_chars']}")
print("\nstatements per label")
print(pilot_df["statements"].describe(percentiles=[.5, .95]).round(1).to_string())

# Handover §3: no chunking needed. ~4 chars/token is the usual English rule of thumb.
est_tok = pilot_df["chars"] / 4
print(f"\nestimated tokens  median {est_tok.median():,.0f}   "
      f"p95 {est_tok.quantile(.95):,.0f}   max {est_tok.max():,.0f}")
print("Comfortably inside any modern context window -> no chunking, as designed.")

### 3.1 Did the `SKIP_TYPES` fix actually land?

Handover §2 required this to be fixed **before** the gold set was generated, because
`Brand IP Statement(s)` and `Seals/Symbols` were being dropped even though Brand IP
carries real benefit claims. If those types are absent from the passages, the gold set
is wrong on exactly those labels and must be regenerated.

In [ ]:
types = Counter()
for r in pilot:
    types.update(re.findall(r"^\[([^\]]+)\]", r["passage"], flags=re.M))

print(f"{len(types)} distinct statement types across {len(pilot):,} labels\n")
print(pd.DataFrame(types.most_common(12), columns=["statement type", "n"]).to_string(index=False))

print("\n--- the two types the bug used to drop ---")
for t in ("Brand IP Statement(s) re: (c), (TM), (SM)", "Seals/Symbols"):
    n_lab = sum(1 for r in pilot if f"[{t}]" in r["passage"])
    print(f"  present on {n_lab:>4} / {len(pilot):,} labels ({n_lab/len(pilot):.0%})   {t}")

print("\n--- the three types that should still be dropped ---")
for t in ("FDA Statement of Identity", "FDA Disclaimer Statement",
          "General: Product/Version Code"):
    print(f"  occurrences: {types.get(t, 0):>4}   {t}")

fixed = all(any(f"[{t}]" in r["passage"] for r in pilot)
            for t in ("Brand IP Statement(s) re: (c), (TM), (SM)", "Seals/Symbols"))
dropped = not any(types.get(t) for t in ("FDA Statement of Identity",
                                         "FDA Disclaimer Statement",
                                         "General: Product/Version Code"))
print(f"\nSKIP_TYPES fix visible in the data: {fixed}")
print(f"Safe types still excluded:           {dropped}")
print("=> gold set was generated post-fix." if fixed and dropped
      else "=> INVESTIGATE before trusting the gold set.")

### 3.2 A whole passage, as the model sees it

This is the exact string an extractor is handed — no preprocessing between this and
the prompt.

In [ ]:
SHOW_ID = 3488   # first gold-labelled row; try 3488 for a dense multi-risk label
print(pilot_by_id[SHOW_ID]["passage"])

## 4. Artifact 2 — the gold extractions

`purpose` and `risk` are lists of `{text, evidence, grounded}` (handover §4), one entry
per distinct claim, plus a per-field 0–5 **extraction confidence** — not a severity
score, deliberately (§1 "explicitly rejected").

In [ ]:
gold_df = pd.DataFrame(
    {"dsld_id": g["dsld_id"], "name": g["name"],
     "n_purpose": len(g["purpose"]), "n_risk": len(g["risk"]),
     "purpose_conf": g["purpose_confidence"], "risk_conf": g["risk_confidence"]}
    for g in gold
)

n_p = int(gold_df.n_purpose.sum())
n_r = int(gold_df.n_risk.sum())
print(f"labels labelled              {len(gold_df)}")
print(f"purpose entries              {n_p:>4}   on {(gold_df.n_purpose > 0).sum()} labels "
      f"({(gold_df.n_purpose > 0).mean():.0%})")
print(f"risk entries                 {n_r:>4}   on {(gold_df.n_risk > 0).sum()} labels "
      f"({(gold_df.n_risk > 0).mean():.0%})")
print(f"labels with neither          {((gold_df.n_purpose == 0) & (gold_df.n_risk == 0)).sum()}")

assert (n_p, n_r) == (gold_mf["n_purpose_entries"], gold_mf["n_risk_entries"]), "manifest disagrees"
print("\ncounts agree with the gold manifest.")

print("\nentries per label")
print(pd.DataFrame({
    "purpose": gold_df.n_purpose.value_counts().sort_index(),
    "risk":    gold_df.n_risk.value_counts().sort_index(),
}).fillna(0).astype(int).rename_axis("entries").to_string())

### 4.1 Confidence — and the recall question it answers

Confidence 0 means *nothing of this field was found on the label*, not *found it
badly*. So the zeros are the empty-label rate, and they are the number that matters
for the whole project: they are the ceiling on how much of the 92,442-row
Purpose/Risk gap this method can close at all.

In [ ]:
conf = pd.DataFrame({
    "purpose": gold_df.purpose_conf.value_counts().sort_index(),
    "risk":    gold_df.risk_conf.value_counts().sort_index(),
}).fillna(0).astype(int).rename_axis("confidence")
conf["purpose %"] = (conf.purpose / len(gold_df) * 100).round(0)
conf["risk %"]    = (conf.risk    / len(gold_df) * 100).round(0)
print(conf.to_string())

for field in ("purpose", "risk"):
    c = gold_df[f"{field}_conf"]
    print(f"\n{field}:  empty (0) {(c == 0).mean():>5.0%}    "
          f"high (4-5) {(c >= 4).mean():>5.0%}    "
          f"mid (1-3) {c.between(1, 3).mean():>5.0%}")

print(f"\n{(gold_df.purpose_conf == 0).sum()} labels state no purpose and "
      f"{(gold_df.risk_conf == 0).sum()} state no risk.")
print(f"On this sample the recoverable fraction is ~{(gold_df.purpose_conf > 0).mean():.0%} "
      f"for Purpose and ~{(gold_df.risk_conf > 0).mean():.0%} for Risk.")
print("Note the asymmetry the per-field score was introduced for: purpose and risk")
print("are absent on *different* labels, so a single per-row score would blur them.")
print(f"  purpose present, risk absent: {((gold_df.n_purpose > 0) & (gold_df.n_risk == 0)).sum()}")
print(f"  risk present, purpose absent: {((gold_df.n_risk > 0) & (gold_df.n_purpose == 0)).sum()}")

### 4.2 Grounding — the fabrication check

Every `evidence` span must appear verbatim in the passage (handover §4, findings §6.3).
The manifest claims `n_ungrounded_entries: 0`; the point of re-checking here is that the
notebook verifies it independently rather than trusting the flag.

In [ ]:
rows, ungrounded = [], []
for g in gold:
    passage = pilot_by_id[g["dsld_id"]]["passage"]
    for field in ("purpose", "risk"):
        for e in g[field]:
            hit = e["evidence"] in passage
            rows.append({"dsld_id": g["dsld_id"], "field": field,
                         "claimed": e.get("grounded"), "verified": hit,
                         "ev_chars": len(e["evidence"]), "text_chars": len(e["text"])})
            if not hit:
                ungrounded.append((g["dsld_id"], field, e["evidence"][:80]))

ev = pd.DataFrame(rows)
print(f"entries checked            {len(ev)}")
print(f"evidence found in passage  {ev.verified.sum()}  ({ev.verified.mean():.1%})")
print(f"disagreements with the stored `grounded` flag: {(ev.claimed != ev.verified).sum()}")
print(f"manifest n_ungrounded_entries: {gold_mf['n_ungrounded_entries']}")

if ungrounded:
    print("\nUNGROUNDED:")
    for u in ungrounded:
        print("  ", u)
else:
    print("\nNo fabricated spans. Independently re-verified, not taken from the flag.")

### 4.3 The known weakness, made visible

Handover §4 states it plainly and this cell quantifies it.

For **purpose**, `text` and `evidence` are near-identical — the check is strong.
For **risk**, DSLD packs several claims into one `notes` block with no substring
isolating one, so `evidence` is the *whole block* and several entries share it. The
check then proves the block is genuine, not that the claim drawn from it is. It catches
wholesale fabrication, not misreading. Bounded and accepted — but a local model scored
against this gold set inherits exactly the same blind spot.

In [ ]:
ratio = (ev.groupby("field")
           .apply(lambda d: pd.Series({
               "entries": len(d),
               "median evidence chars": d.ev_chars.median(),
               "median text chars": d.text_chars.median(),
               "evidence/text size ratio": round(d.ev_chars.median() / d.text_chars.median(), 1),
           }), include_groups=False))
print(ratio.to_string())

shared = 0
for g in gold:
    counts = Counter(e["evidence"] for e in g["risk"])
    shared += sum(n for n in counts.values() if n > 1)
print(f"\nrisk entries sharing an evidence block with a sibling: {shared} / {n_r} "
      f"({shared / n_r:.0%})")
print("For those, `evidence in passage` verifies the block, not the individual claim.")

worst = max(gold, key=lambda g: max(Counter(e["evidence"] for e in g["risk"]).values(), default=0))
print(f"\n--- worst case: label {worst['dsld_id']} ({worst['name']}) ---")
print("one evidence block:")
print(textwrap.fill(worst["risk"][0]["evidence"], 92, initial_indent="  ", subsequent_indent="  "))
print(f"\nsplit into {len(worst['risk'])} separate risk claims:")
for e in worst["risk"]:
    print(f"  - {e['text']}")

### 4.4 Was boilerplate actually suppressed?

Findings §9 flags the dilution problem: a scalar `Risk` string swallows "keep out of
reach of children", "store in a cool dry place" and similar, and it cannot be undone
afterwards. Per-entry rows were chosen so a suppression pass has somewhere to act — but
that pass is still undesigned (§9 "still open"). So: did the human-directed labelling
suppress it by hand?

In [ ]:
BOILERPLATE = ["keep out of reach", "store in a cool", "do not use if seal",
               "tamper", "keep away from children", "safety seal"]

in_passage = sum(1 for g in gold
                 if any(b in pilot_by_id[g["dsld_id"]]["passage"].lower() for b in BOILERPLATE))
as_risk = [(g["dsld_id"], e["text"]) for g in gold for e in g["risk"]
           if any(b in e["text"].lower() for b in BOILERPLATE)]

print(f"gold labels whose passage contains boilerplate caution text: {in_passage} / {len(gold)}")
print(f"boilerplate that leaked into a `risk` entry:                 {len(as_risk)} / {n_r}")
if as_risk:
    for r in as_risk[:10]:
        print("  ", r)
else:
    print("\nNone. The gold set encodes the intended behaviour: boilerplate is not a risk.")
    print("It is an implicit spec a local model will be scored against, but it is nowhere")
    print("written down as a rule -- worth stating in the prompt rather than hoping the")
    print("model infers it from the gold set.")

### 4.5 Read some actual extractions

Numbers stop being useful here. This is the part to read.

In [ ]:
def show(dsld_id, passage=False):
    g = next(x for x in gold if x["dsld_id"] == dsld_id)
    print("=" * 96)
    print(f"{g['dsld_id']}  {g['name']}   "
          f"[purpose conf {g['purpose_confidence']} | risk conf {g['risk_confidence']}]")
    if passage:
        print("-" * 96)
        print(textwrap.indent(pilot_by_id[dsld_id]["passage"], "  "))
    for field in ("purpose", "risk"):
        print("-" * 96)
        if not g[field]:
            print(f"{field.upper()}: (none on this label)")
            continue
        print(f"{field.upper()}:")
        for e in g[field]:
            print(f"  * {e['text']}")
            print(textwrap.fill(f"evidence: {e['evidence']}", 92,
                                initial_indent="      ", subsequent_indent="      "))

# a clean two-purpose label, a dense multi-risk one, and one with nothing to extract
show(709)
show(3488)
empty = gold_df[(gold_df.n_purpose == 0) & (gold_df.n_risk == 0)]
show(int(empty.iloc[0].dsld_id), passage=True)

## 5. Is the gold set a fair sample of the pilot?

The gold 100 were meant to be 100 *of* the 1,000 (handover §6). This cell checks whether
they are representative, because every future comparison of a local model rests on it.

In [ ]:
ids = [r["dsld_id"] for r in pilot]
gold_ids = [g["dsld_id"] for g in gold]

print(f"pilot stored in ascending id order: {ids == sorted(ids)}")
print(f"gold == the first 100 pilot rows:   {gold_ids == ids[:100]}")
print(f"\ngold id range   {min(gold_ids):>7,} .. {max(gold_ids):>7,}")
print(f"pilot id range  {min(ids):>7,} .. {max(ids):>7,}")
print(f"gold covers the bottom {sum(i <= max(gold_ids) for i in ids) / len(ids):.0%} "
      f"of the pilot by id")

rest = pilot_df[~pilot_df.dsld_id.isin(gold_ids)]
head = pilot_df[pilot_df.dsld_id.isin(gold_ids)]
print("\npassage chars       gold-100   other-900")
print(f"  median          {head.chars.median():>9,.0f} {rest.chars.median():>11,.0f}")
print(f"  p95             {head.chars.quantile(.95):>9,.0f} {rest.chars.quantile(.95):>11,.0f}")
print("\nstatements/label    gold-100   other-900")
print(f"  median          {head.statements.median():>9,.1f} {rest.statements.median():>11,.1f}")

> **Finding — the gold 100 are not a random draw.**
>
> The pilot file is sorted by `dsld_id`, and the gold set is its first 100 rows: IDs
> 709–20287, the lowest-numbered ~10% of the sample. DSLD IDs are issued over time, so
> this is an **age-biased slice** — the oldest labels in the sample, not a cross-section
> of it. The 1,000-row *pilot* is properly randomised (seeded, recorded in the manifest);
> only the 100 selected from it are not.
>
> Passage length and statement count are close between the two groups, so it is not
> grossly unrepresentative, and nothing already labelled is wasted. But label conventions
> drift with time — claim phrasing, warning boilerplate and Structure/Function language
> have all changed since the early DSLD entries — so scores measured on it may not
> transfer to the full 92,442 rows.
>
> Cheap fix when the gold set is next extended: draw the additional labels randomly from
> the remaining 900 with the manifest's seed, rather than taking the next 100 in order.
> Worth doing before any local-model score is treated as a corpus-wide number.

## 6. Summary

What the two artifacts establish, and what they do not.

In [ ]:
brand_n = sum(1 for r in pilot if "[Brand IP" in r["passage"])
seals_n = sum(1 for r in pilot if "[Seals/Symbols]" in r["passage"])
both_empty = int(((gold_df.n_purpose == 0) & (gold_df.n_risk == 0)).sum())

L = []
L.append(f"ARTIFACT 1 -- flattened labels           {len(pilot):,} rows")
L.append(f"  median {pilot_df.chars.median():,.0f} chars (~{pilot_df.chars.median()/4:,.0f} tokens), "
         f"max {pilot_df.chars.max():,} -- no chunking needed")
L.append(f"  SKIP_TYPES fix visible: Brand IP on {brand_n} labels, Seals/Symbols on {seals_n}")
L.append(f"  dump coverage shortfall {pilot_mf['n_ids_missing_from_dump']} / "
         f"{pilot_mf['n_ids_referenced']:,} ids -- noise")

L.append("")
L.append(f"ARTIFACT 2 -- gold extractions           {len(gold)} rows")
L.append(f"  purpose on {(gold_df.n_purpose > 0).sum()}/100 labels ({n_p} entries), "
         f"risk on {(gold_df.n_risk > 0).sum()}/100 ({n_r} entries)")
L.append(f"  {both_empty} labels state neither -- a real ceiling on what this method can recover")
L.append(f"  all {len(ev)} evidence spans verified against the passage; {len(ungrounded)} fabricated")
L.append(f"  boilerplate leaked into risk: {len(as_risk)}")

L.append("")
L.append("CARRIED FORWARD")
L.append(f"  - risk evidence is block-level: {shared}/{n_r} entries share a block with a sibling,")
L.append("    so grounding proves the block, not the claim. A local model inherits this blind spot.")
L.append("  - the gold 100 are the lowest-id 100 of the pilot, not a random draw (section 5).")
L.append("  - boilerplate suppression is in the gold data as behaviour but not written as a rule.")

L.append("")
L.append("STILL OPEN (handover section 9) -- untouched by this notebook")
L.append("  - which local model and quant; no weights on disk")
L.append("  - GBNF grammar for the list-of-objects schema")
L.append("  - whether to split purpose and risk into two calls")
L.append("  - column naming for DSLD-derived values (must not merge into LNHPD's columns)")
L.append(f"  - the {pilot_mf['n_both_rows_held_back']} held-back \"both\" rows remain unspent -- "
         "do not burn them on prompt iteration")

print("\n".join(L))